In [4]:
import pandas as pd
import numpy as np

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [5]:
train = pd.read_csv('/content/train.csv', low_memory=False)
store = pd.read_csv('/content/store.csv')

print("Train shape:", train.shape)
print("Store shape:", store.shape)

Train shape: (1017209, 9)
Store shape: (1115, 10)


In [6]:
print("Rows:", len(train))
print("Columns:", train.columns.tolist())

print("Date range:")
print(train['Date'].min(), train['Date'].max())

print("Unique stores:", train['Store'].nunique())

Rows: 1017209
Columns: ['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']
Date range:
2013-01-01 2015-07-31
Unique stores: 1115


In [7]:
train = pd.merge(train, store, on='Store', how='left')

print("Merged shape:", train.shape)

Merged shape: (1017209, 18)


In [8]:
train['Date'] = pd.to_datetime(train['Date'])

train['Month'] = train['Date'].dt.month
train['Day'] = train['Date'].dt.day

In [9]:
train['CompetitionDistance'] = train['CompetitionDistance'].fillna(
    train['CompetitionDistance'].median()
)

train['CompetitionOpenSinceMonth'] = train['CompetitionOpenSinceMonth'].fillna(0)
train['CompetitionOpenSinceYear'] = train['CompetitionOpenSinceYear'].fillna(0)

train['Promo2SinceWeek'] = train['Promo2SinceWeek'].fillna(0)
train['Promo2SinceYear'] = train['Promo2SinceYear'].fillna(0)

train['PromoInterval'] = train['PromoInterval'].fillna('NoPromo2')

In [10]:
print(train.isnull().sum().sum())

0


In [11]:
categorical_cols = [
    'StateHoliday',
    'StoreType',
    'Assortment',
    'PromoInterval'
]

train = pd.get_dummies(
    train,
    columns=categorical_cols,
    dtype=int
)

print(train.shape)

(1017209, 31)


In [12]:
X = train.drop(columns=['Sales', 'Customers', 'Date'])
y = train['Sales']

In [13]:
train = train.sort_values('Date')

split_date = train['Date'].max() - pd.Timedelta(days=42)

train_data = train[train['Date'] <= split_date]
val_data = train[train['Date'] > split_date]

X_train = train_data.drop(columns=['Sales', 'Customers', 'Date'])
y_train = train_data['Sales']

X_val = val_data.drop(columns=['Sales', 'Customers', 'Date'])
y_val = val_data['Sales']

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)

Training shape: (970379, 28)
Validation shape: (46830, 28)


In [14]:
X_tune = X_train.iloc[-200000:]
y_tune = y_train.loc[X_tune.index]

print("X_tune:", X_tune.shape)
print("y_tune:", y_tune.shape)

X_tune: (200000, 28)
y_tune: (200000,)


In [15]:
bagging_model = BaggingRegressor(
    estimator=DecisionTreeRegressor(
        max_depth=30,
        min_samples_split=13,
        min_samples_leaf=2,
        random_state=42
    ),
    n_estimators=50,
    max_samples=0.8,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

bagging_model.fit(X_tune, y_tune)

BaggingRegressor(estimator=DecisionTreeRegressor(max_depth=30,
                                                 min_samples_leaf=2,
                                                 min_samples_split=13,
                                                 random_state=42),
                 max_samples=0.8, n_estimators=50, n_jobs=-1, random_state=42)

In [16]:
train_pred = bagging_model.predict(X_tune)
val_pred = bagging_model.predict(X_val)

print("Training R2:", r2_score(y_tune, train_pred))
print("Validation R2:", r2_score(y_val, val_pred))

print("MAE:", mean_absolute_error(y_val, val_pred))

rmse = np.sqrt(mean_squared_error(y_val, val_pred))
print("RMSE:", rmse)

Training R2: 0.9599768483609907
Validation R2: 0.8879677997879196
MAE: 815.2433892453456
RMSE: 1247.171363303046


In [17]:
print(train.shape)
print(train['Date'].min(), train['Date'].max())

(1017209, 31)
2013-01-01 00:00:00 2015-07-31 00:00:00


In [18]:
train = pd.merge(train, store, on='Store', how='left')

print(train.shape)

(1017209, 40)


In [19]:
print(train.columns.tolist())
print(train.shape)

['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'SchoolHoliday', 'CompetitionDistance_x', 'CompetitionOpenSinceMonth_x', 'CompetitionOpenSinceYear_x', 'Promo2_x', 'Promo2SinceWeek_x', 'Promo2SinceYear_x', 'Month', 'Day', 'StateHoliday_0', 'StateHoliday_a', 'StateHoliday_b', 'StateHoliday_c', 'StoreType_a', 'StoreType_b', 'StoreType_c', 'StoreType_d', 'Assortment_a', 'Assortment_b', 'Assortment_c', 'PromoInterval_Feb,May,Aug,Nov', 'PromoInterval_Jan,Apr,Jul,Oct', 'PromoInterval_Mar,Jun,Sept,Dec', 'PromoInterval_NoPromo2', 'StoreType', 'Assortment', 'CompetitionDistance_y', 'CompetitionOpenSinceMonth_y', 'CompetitionOpenSinceYear_y', 'Promo2_y', 'Promo2SinceWeek_y', 'Promo2SinceYear_y', 'PromoInterval']
(1017209, 40)


In [20]:
train = pd.read_csv('/content/train.csv', low_memory=False)
store = pd.read_csv('/content/store.csv')

train = pd.merge(train, store, on='Store', how='left')

In [21]:
print(train.shape)
print(train.columns.tolist())

(1017209, 18)
['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval']


In [22]:
train['Date'] = pd.to_datetime(train['Date'])

train['Month'] = train['Date'].dt.month
train['Day'] = train['Date'].dt.day

In [23]:
train['CompetitionDistance'] = train['CompetitionDistance'].fillna(
    train['CompetitionDistance'].median()
)

train['CompetitionOpenSinceMonth'] = train['CompetitionOpenSinceMonth'].fillna(0)
train['CompetitionOpenSinceYear'] = train['CompetitionOpenSinceYear'].fillna(0)

train['Promo2SinceWeek'] = train['Promo2SinceWeek'].fillna(0)
train['Promo2SinceYear'] = train['Promo2SinceYear'].fillna(0)

train['PromoInterval'] = train['PromoInterval'].fillna('NoPromo2')

In [24]:
categorical_cols = [
    'StateHoliday',
    'StoreType',
    'Assortment',
    'PromoInterval'
]

train = pd.get_dummies(
    train,
    columns=categorical_cols,
    dtype=int
)

In [25]:
print("Final shape:", train.shape)
print("Missing values:", train.isnull().sum().sum())
print(train.head())

Final shape: (1017209, 31)
Missing values: 0
   Store  DayOfWeek       Date  Sales  Customers  Open  Promo  SchoolHoliday  \
0      1          5 2015-07-31   5263        555     1      1              1   
1      2          5 2015-07-31   6064        625     1      1              1   
2      3          5 2015-07-31   8314        821     1      1              1   
3      4          5 2015-07-31  13995       1498     1      1              1   
4      5          5 2015-07-31   4822        559     1      1              1   

   CompetitionDistance  CompetitionOpenSinceMonth  ...  StoreType_b  \
0               1270.0                        9.0  ...            0   
1                570.0                       11.0  ...            0   
2              14130.0                       12.0  ...            0   
3                620.0                        9.0  ...            0   
4              29910.0                        4.0  ...            0   

   StoreType_c  StoreType_d  Assortment_a  Asso

In [26]:
X = train.drop(columns=['Sales', 'Customers', 'Date'])
y = train['Sales']

In [27]:
train = train.sort_values('Date')

split_date = train['Date'].max() - pd.Timedelta(days=42)

train_data = train[train['Date'] <= split_date]
val_data = train[train['Date'] > split_date]

X_train = train_data.drop(columns=['Sales', 'Customers', 'Date'])
y_train = train_data['Sales']

X_val = val_data.drop(columns=['Sales', 'Customers', 'Date'])
y_val = val_data['Sales']

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)

Training shape: (970379, 28)
Validation shape: (46830, 28)


In [28]:
X_tune = X_train.iloc[-200000:]
y_tune = y_train.loc[X_tune.index]

print(X_tune.shape)
print(y_tune.shape)

(200000, 28)
(200000,)


In [29]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor

bagging_model = BaggingRegressor(
    estimator=DecisionTreeRegressor(
        max_depth=30,
        min_samples_split=13,
        min_samples_leaf=2,
        random_state=42
    ),
    n_estimators=50,
    max_samples=0.8,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

bagging_model.fit(X_tune, y_tune)

BaggingRegressor(estimator=DecisionTreeRegressor(max_depth=30,
                                                 min_samples_leaf=2,
                                                 min_samples_split=13,
                                                 random_state=42),
                 max_samples=0.8, n_estimators=50, n_jobs=-1, random_state=42)

In [30]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

train_pred = bagging_model.predict(X_tune)
val_pred = bagging_model.predict(X_val)

print("Training R2:", r2_score(y_tune, train_pred))
print("Validation R2:", r2_score(y_val, val_pred))

print("MAE:", mean_absolute_error(y_val, val_pred))

rmse = np.sqrt(mean_squared_error(y_val, val_pred))
print("RMSE:", rmse)

Training R2: 0.9599768483609907
Validation R2: 0.8879677997879196
MAE: 815.2433892453456
RMSE: 1247.171363303046


In [31]:
test = pd.read_csv('/content/test.csv')

print(test.shape)
test.head()

(41088, 8)


,Id,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday
0,1,1,4,2015-09-17,1.0,1,0,0
1,2,3,4,2015-09-17,1.0,1,0,0
2,3,7,4,2015-09-17,1.0,1,0,0
3,4,8,4,2015-09-17,1.0,1,0,0
4,5,9,4,2015-09-17,1.0,1,0,0


In [32]:
test_ids = test['Id'].copy()

test = pd.merge(test, store, on='Store', how='left')

In [33]:
test['Date'] = pd.to_datetime(test['Date'])

test['Month'] = test['Date'].dt.month
test['Day'] = test['Date'].dt.day

In [34]:
test['CompetitionDistance'] = test['CompetitionDistance'].fillna(
    train['CompetitionDistance'].median()
)

test['CompetitionOpenSinceMonth'] = test['CompetitionOpenSinceMonth'].fillna(0)
test['CompetitionOpenSinceYear'] = test['CompetitionOpenSinceYear'].fillna(0)

test['Promo2SinceWeek'] = test['Promo2SinceWeek'].fillna(0)
test['Promo2SinceYear'] = test['Promo2SinceYear'].fillna(0)

test['PromoInterval'] = test['PromoInterval'].fillna('NoPromo2')

test['Open'] = test['Open'].fillna(1)

In [35]:
categorical_cols = [
    'StateHoliday',
    'StoreType',
    'Assortment',
    'PromoInterval'
]

test = pd.get_dummies(
    test,
    columns=categorical_cols,
    dtype=int
)

In [36]:
X_test = test.drop(columns=['Id', 'Date'])

X_test = X_test.reindex(
    columns=X_tune.columns,
    fill_value=0
)

print(X_test.shape)
print(X_tune.shape)

(41088, 28)
(200000, 28)


In [37]:
predictions = bagging_model.predict(X_test)

predictions = np.maximum(predictions, 0)

predictions[test['Open'].to_numpy() == 0] = 0

print(predictions[:10])

[4463.32940701 8402.18688605 9380.95081134 7531.08463874 8299.98943745
 6204.62883769 7337.32301712 8838.13662983 5795.64272275 6313.32324357]


In [38]:
submission = pd.DataFrame({
    'Id': test_ids,
    'Sales': predictions
})

print(submission.head())
print(submission.shape)

   Id        Sales
0   1  4463.329407
1   2  8402.186886
2   3  9380.950811
3   4  7531.084639
4   5  8299.989437
(41088, 2)


In [39]:
submission.to_csv('/content/submission.csv', index=False)